# VizEx — image feature extraction

Single experiment: participant recreations (trials) plus the target images shown
during validation trials.

Run this notebook first, then `030_feature_validation_analysis`, then
`035_feature_experimental_analysis`.

**Outputs written to `015_tables/`**
- `vizex_trials.csv` — one row per cropped trial image
- `targs.csv` — one row per target image
- `vizex_features.csv`, `vizex_targets_features.csv` — the above plus feature columns

Every table is rewritten from scratch on each run. Nothing merges into existing files.

## Setup

In [ ]:
## imports

import os
import math
import random
import shutil
import colorsys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter, gaussian_laplace, sobel

In [ ]:
## paths

PROJECT_PATH = Path().resolve().parent
TABLES_PATH = PROJECT_PATH / "015_tables"
TRIAL_IMG_DIR = PROJECT_PATH / "005_cleaned_data"
TARGET_IMG_DIR = PROJECT_PATH / "007_targets"
VIZ_PATH = PROJECT_PATH / "020_visualizations"

TABLES_PATH.mkdir(parents=True, exist_ok=True)
VIZ_PATH.mkdir(parents=True, exist_ok=True)

print(PROJECT_PATH)

In [ ]:
## configuration
#
# Every tunable constant lives here so the pipeline has no hidden settings.

# Preprocessing
CROP_PIXELS = 12          # fixed border (alpha padding) removed from all four sides
EDGE_BLUR_SIGMA = 1.0     # Gaussian blur applied before edge features

# Features
DETAIL_LOG_SIGMA = 1.0    # Laplacian-of-Gaussian sigma for the detail feature
ORI_KAPPA = 8.0           # angular selectivity of the orientation kernels
ORI_EPS = 1e-8
NORM_QUANTILE = 0.95      # raw value at this quantile is mapped to 1.0

# Heatmaps
N_BINS = 35
TRIM_Q = 0.10             # symmetric trim for the robust across-image reducer
HEATMAP_FIGSIZE = (3.0, 3.0)
SCALE_FIGSIZE = (0.6, 3.0)
DPI = 300
SAVE_PREFIX = "hm"

# Conditions. Numeric codes are strobe frequencies; A00-C14 are validation images.
HZ_MAP = {1: 2.5, 2: 5, 3: 10, 4: 15, 5: 20, 6: 40, 7: 80}
CONTROL_CODE = "7"        # 80 Hz control
EXPERIMENTAL_CONDS = tuple(str(k) for k in HZ_MAP)

# Table filenames
TRIALS_CSV = TABLES_PATH / "vizex_trials.csv"
TARGETS_CSV = TABLES_PATH / "targs.csv"
TRIALS_FEATURES_CSV = TABLES_PATH / "vizex_features.csv"
TARGETS_FEATURES_CSV = TABLES_PATH / "vizex_targets_features.csv"

## Build the trial and target tables

In [ ]:
## trial table: cleaned_file_index.csv -> vizex_trials.csv

def _format_hz(hz) -> str:
    """5.0 -> '5', 2.5 -> '2.5'."""
    hz = float(hz)
    return str(int(hz)) if hz.is_integer() else str(hz)


def _condition_code_to_int(x):
    """Integer strobe code, or None for validation image codes like 'C14'."""
    if pd.isna(x):
        return None
    try:
        f = float(str(x).strip())
    except ValueError:
        return None
    return int(f) if np.isfinite(f) and float(f).is_integer() else None


index_fp = TABLES_PATH / "cleaned_file_index.csv"
src = pd.read_csv(index_fp)

trials = pd.DataFrame({
    "participant_code": src["final_participant"],
    "condition_code": src["trialT"].astype(str).str.strip(),
    "trialn": pd.to_numeric(src["trialN"], errors="coerce").astype("Int64"),
    "png_filename": src["clean_filename"].astype(str).str.strip(),
})

# Drop non-trial rows
trials = trials[~trials["condition_code"].isin(["9", "POSTQ"])].copy()

# Keep ONLY the cropped render of each drawing.
#
# The uncropped PNGs are backups of the original and must never enter the analysis.
# Do not test this with .str.endswith("cropped.png") -- "..._fileType_uncropped.png"
# also ends with "cropped.png", which previously let every backup through and
# doubled the dataset, contaminating the normalisation quantiles.
trials = trials[trials["png_filename"].str.contains("fileType_cropped", na=False)].copy()

leaked = trials["png_filename"].str.contains("uncropped", na=False)
if leaked.any():
    raise ValueError(
        f"{int(leaked.sum())} uncropped rows survived the filter, e.g. "
        f"{trials.loc[leaked, 'png_filename'].iloc[0]}"
    )

# Derive condition metadata.
#
# Keep this as a plain Python list, not a pandas Series. Mapping through a Series
# coerces the integer codes to float and None to NaN, which silently breaks the
# `is None` and `== CONTROL_CODE` tests below.
code_ints = [_condition_code_to_int(c) for c in trials["condition_code"]]


def _condition_type(ci) -> str:
    if ci is None:
        return "VALIDATION"          # a target image was shown
    if str(ci) == CONTROL_CODE:
        return "CONTROL"             # 80 Hz
    if 1 <= ci <= 6:
        return "EXPERIMENTAL"
    raise ValueError(f"Unexpected numeric condition code: {ci}")


trials["stimulus_type"] = ["IMAGE" if ci is None else "LIGHT" for ci in code_ints]
trials["condition_hz"] = pd.array(
    [None if ci is None else HZ_MAP[ci] for ci in code_ints], dtype="Float64")
trials["condition_name"] = [
    None if ci is None else f"{_format_hz(HZ_MAP[ci])} Hz" for ci in code_ints]
trials["condition_type"] = [_condition_type(ci) for ci in code_ints]

# Every numeric code must be a known frequency, and the three labels must agree.
assert set(trials["condition_type"]) <= {"VALIDATION", "CONTROL", "EXPERIMENTAL"}
assert (trials.loc[trials["stimulus_type"] == "IMAGE", "condition_hz"].isna()).all()
assert (trials.loc[trials["stimulus_type"] == "LIGHT", "condition_hz"].notna()).all()

trials = trials[[
    "participant_code", "condition_code", "condition_name", "condition_hz",
    "condition_type", "stimulus_type", "trialn", "png_filename",
]].reset_index(drop=True)

# png_filename is the merge key everywhere downstream, so it must be unique.
dupes = trials["png_filename"].duplicated(keep=False)
if dupes.any():
    raise ValueError(
        f"{int(dupes.sum())} duplicated png_filename values, e.g. "
        f"{trials.loc[dupes, 'png_filename'].iloc[0]}"
    )

trials.to_csv(TRIALS_CSV, index=False)
print(f"Saved {TRIALS_CSV.name}: {len(trials):,} rows (cropped only)")
print(f"  participants: {trials['participant_code'].nunique()}")
print(f"  conditions:   {sorted(trials['condition_code'].unique())}")
print(f"  LIGHT / IMAGE: {(trials['stimulus_type'] == 'LIGHT').sum()} / "
      f"{(trials['stimulus_type'] == 'IMAGE').sum()}")
print("  condition_type:")
print(trials["condition_type"].value_counts().to_string().replace("\n", "\n    "))

In [ ]:
## target table: A00-A14, B00-B14, C00-C14

targets = pd.DataFrame([
    {"condition_code": f"{p}{i:02d}",
     "condition_name": f"Image {p}{i:02d}",
     "png_filename": f"{p}{i:02d}.png"}
    for p in ("A", "B", "C") for i in range(15)
])

targets.to_csv(TARGETS_CSV, index=False)
print(f"Saved {TARGETS_CSV.name}: {len(targets)} rows")

In [ ]:
## load both tables and verify every image resolves on disk

df_trials = pd.read_csv(TRIALS_CSV)
df_targets = pd.read_csv(TARGETS_CSV)

df_trials["condition_code"] = df_trials["condition_code"].astype(str)
df_targets["condition_code"] = df_targets["condition_code"].astype(str)


def check_images_exist(df: pd.DataFrame, img_dir: Path, label: str) -> None:
    """Fail loudly rather than producing a silently-NaN feature column."""
    if not img_dir.is_dir():
        raise NotADirectoryError(f"[{label}] image directory not found: {img_dir}")

    names = df["png_filename"].dropna().astype(str)
    missing = [n for n in names if not (img_dir / n).exists()]
    if missing:
        raise FileNotFoundError(
            f"[{label}] {len(missing)} of {len(names)} images not found under {img_dir}. "
            f"First few: {missing[:3]}"
        )
    print(f"[{label}] {len(names)} rows, all images resolved on disk.")


check_images_exist(df_trials, TRIAL_IMG_DIR, "trials")
check_images_exist(df_targets, TARGET_IMG_DIR, "targets")

display(df_trials.head())
display(df_targets.head())

## Preprocessing

In [ ]:
## image loading and the two preprocessing pipelines

def load_grayscale(path) -> np.ndarray:
    """Load any image as float32 grayscale in [0, 1]."""
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0


def crop_border(arr: np.ndarray, crop_pixels: int = CROP_PIXELS) -> np.ndarray:
    """Remove a fixed border of known width from all four sides."""
    if crop_pixels <= 0:
        return arr
    h, w = arr.shape[:2]
    if 2 * crop_pixels >= min(h, w):
        raise ValueError(f"crop_pixels={crop_pixels} too large for image shape {arr.shape}")
    return arr[crop_pixels:h - crop_pixels, crop_pixels:w - crop_pixels]


def minmax01(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    mn, mx = float(np.nanmin(arr)), float(np.nanmax(arr))
    return (arr - mn) / (mx - mn + 1e-8)


def preprocess_edges(img: np.ndarray) -> np.ndarray:
    """For edge-based features: crop, contrast-normalise, blur."""
    arr = crop_border(img.astype(np.float32, copy=False))
    return gaussian_filter(minmax01(arr), sigma=EDGE_BLUR_SIGMA)


def preprocess_intensity(img: np.ndarray) -> np.ndarray:
    """For intensity-based features: crop only. No blur, no normalisation, so
    absolute brightness is preserved."""
    return crop_border(img.astype(np.float32, copy=False))

## Feature maps

Every feature is a function `(preprocessed_image) -> float32 heatmap`. A separate
reducer turns the heatmap into the per-image scalar, so the same function serves
both the scalar tables and the spatial heatmaps.

In [ ]:
## detail: edge density via |Laplacian of Gaussian|

def detail_map(pre: np.ndarray) -> np.ndarray:
    return np.abs(gaussian_laplace(pre.astype(np.float32, copy=False),
                                   sigma=DETAIL_LOG_SIGMA)).astype(np.float32, copy=False)

In [ ]:
## radial and tangential edge orientation, about the image centre
#
# Edge orientation is PERPENDICULAR to the intensity gradient. A radially oriented
# edge therefore has a tangentially aligned gradient, which is why the radial
# feature is built from A_t and the tangential feature from A_r. This is
# intentional -- do not "fix" it by swapping the projections.

def _radial_tangential_unit_vectors(h: int, w: int):
    """Unit radial and tangential vectors about the image centre."""
    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
    y, x = np.mgrid[0:h, 0:w]
    dx = (x - cx).astype(np.float32)
    dy = (y - cy).astype(np.float32)
    r = np.sqrt(dx * dx + dy * dy).astype(np.float32) + ORI_EPS
    ur_x, ur_y = dx / r, dy / r
    return ur_x, ur_y, -ur_y, ur_x  # ur_x, ur_y, ut_x, ut_y


def _sobel_gradients(img: np.ndarray):
    gx = sobel(img, axis=1, mode="reflect").astype(np.float32, copy=False)
    gy = sobel(img, axis=0, mode="reflect").astype(np.float32, copy=False)
    return gx, gy


def _soft_directionality(pre: np.ndarray):
    """Soft radial/tangential gradient activations and the gradient magnitude.

        A_* = |g| * exp(-kappa * (1 - c_*)),  c_* = |g . u_*| / |g|
    """
    img = pre.astype(np.float32, copy=False)
    h, w = img.shape[:2]
    ur_x, ur_y, ut_x, ut_y = _radial_tangential_unit_vectors(h, w)

    gx, gy = _sobel_gradients(img)
    mag = np.hypot(gx, gy).astype(np.float32, copy=False)

    c_r = np.abs(gx * ur_x + gy * ur_y) / (mag + ORI_EPS)
    c_t = np.abs(gx * ut_x + gy * ut_y) / (mag + ORI_EPS)

    A_r = mag * np.exp(-ORI_KAPPA * (1.0 - c_r)).astype(np.float32, copy=False)
    A_t = mag * np.exp(-ORI_KAPPA * (1.0 - c_t)).astype(np.float32, copy=False)
    return A_r.astype(np.float32), A_t.astype(np.float32), mag


def r_directionality_map(pre: np.ndarray) -> np.ndarray:
    """Radial edge orientation: mean of this map is mean(A_t) / mean(|g|)."""
    _, A_t, mag = _soft_directionality(pre)
    denom = float(np.mean(mag[np.isfinite(mag)]) + ORI_EPS) if mag.size else ORI_EPS
    return (A_t / denom).astype(np.float32, copy=False)


def theta_directionality_map(pre: np.ndarray) -> np.ndarray:
    """Tangential edge orientation: mean of this map is mean(A_r) / mean(|g|)."""
    A_r, _, mag = _soft_directionality(pre)
    denom = float(np.mean(mag[np.isfinite(mag)]) + ORI_EPS) if mag.size else ORI_EPS
    return (A_r / denom).astype(np.float32, copy=False)


def sobel_radial_tangential_magnitude(pre: np.ndarray):
    """Unweighted directional edge energy, used for the directionality heatmaps.

        G_r (radial edge)     = |g . u_t|
        G_t (tangential edge) = |g . u_r|

    Satisfies sqrt(G_r^2 + G_t^2) == |g|, so Sobel magnitude is recoverable.
    """
    img = pre.astype(np.float32, copy=False)
    h, w = img.shape
    ur_x, ur_y, ut_x, ut_y = _radial_tangential_unit_vectors(h, w)
    gx, gy = _sobel_gradients(img)
    g_r = np.abs(gx * ut_x + gy * ut_y).astype(np.float32, copy=False)
    g_t = np.abs(gx * ur_x + gy * ur_y).astype(np.float32, copy=False)
    return g_r, g_t

In [ ]:
## brightness and contrast

def brightness_map(pre: np.ndarray) -> np.ndarray:
    """Pixel intensity itself; the mean is overall brightness."""
    return pre.astype(np.float32, copy=False)


def contrast_map(pre: np.ndarray) -> np.ndarray:
    """Squared deviation from the image mean. sqrt(mean(map)) is RMS contrast."""
    img = pre.astype(np.float32, copy=False)
    mu = float(np.mean(img[np.isfinite(img)])) if img.size else 0.0
    return (img - mu) ** 2

In [ ]:
## map -> scalar reducers, and the feature registry

def value_mean(m: np.ndarray) -> float:
    if m is None or m.size == 0:
        return 0.0
    v = m[np.isfinite(m)]
    return float(v.mean()) if v.size else 0.0


def value_rms(m: np.ndarray) -> float:
    """RMS from a squared-deviation map: sqrt(mean((I - mean(I))^2))."""
    if m is None or m.size == 0:
        return 0.0
    v = m[np.isfinite(m)]
    return float(np.sqrt(max(v.mean(), 0.0))) if v.size else 0.0


# normalise=True  -> scale so that Q95 of the TRIAL raw values maps to 1.0
# normalise=False -> raw values are already interpretable, scale is fixed at 1.0
FEATURES = {
    "detail": {
        "label": "Normalised detail level",
        "preprocess": preprocess_edges,
        "map_fn": detail_map,
        "value_fn": value_mean,
        "normalise": True,
    },
    "r_directionality": {
        "label": "Normalised radial orientation",
        "preprocess": preprocess_edges,
        "map_fn": r_directionality_map,
        "value_fn": value_mean,
        "normalise": True,
    },
    "theta_directionality": {
        "label": "Normalised tangential orientation",
        "preprocess": preprocess_edges,
        "map_fn": theta_directionality_map,
        "value_fn": value_mean,
        "normalise": True,
    },
    "brightness": {
        "label": "Mean image brightness (0-1)",
        "preprocess": preprocess_intensity,
        "map_fn": brightness_map,
        "value_fn": value_mean,
        "normalise": False,
    },
    "contrast": {
        "label": "Mean image contrast (RMS)",
        "preprocess": preprocess_intensity,
        "map_fn": contrast_map,
        "value_fn": value_rms,
        "normalise": False,
    },
}

FEATURE_NAMES = list(FEATURES)

## Compute the features

In [ ]:
## per-image computation

def compute_raw_values(df: pd.DataFrame, img_dir: Path, feature: str, label: str) -> np.ndarray:
    """Raw per-image scalar for one feature, aligned to df row order."""
    spec = FEATURES[feature]
    pre_fn, map_fn, val_fn = spec["preprocess"], spec["map_fn"], spec["value_fn"]

    values = np.full(len(df), np.nan, dtype=np.float32)
    missing = []

    for i, fname in enumerate(df["png_filename"].astype(str).tolist()):
        path = img_dir / fname
        if not path.exists():
            missing.append(fname)
            continue
        values[i] = val_fn(map_fn(pre_fn(load_grayscale(path))))

    if missing:
        raise FileNotFoundError(
            f"[{label} | {feature}] {len(missing)} images missing under {img_dir}. "
            f"First few: {missing[:3]}"
        )

    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise ValueError(f"[{label} | {feature}] produced no finite values.")
    if finite.size < len(df):
        print(f"  WARNING [{label} | {feature}]: {len(df) - finite.size} non-finite values.")
    return values


def add_feature(feature: str) -> float:
    """Compute `feature` for trials and targets, writing `<feature>_raw` and
    `<feature>`. The normalisation scale is derived from the TRIALS only and
    reused for the targets, so the two are on a common scale.

    Returns the scale used.
    """
    raw_trials = compute_raw_values(df_trials, TRIAL_IMG_DIR, feature, "trials")
    raw_targets = compute_raw_values(df_targets, TARGET_IMG_DIR, feature, "targets")

    if FEATURES[feature]["normalise"]:
        q = float(np.nanquantile(raw_trials[np.isfinite(raw_trials)], NORM_QUANTILE))
        scale = 1.0 / q if np.isfinite(q) and q > 0 else 1.0
        print(f"[{feature}] norm_scale (Q{NORM_QUANTILE:.2f} -> 1): {scale:.6g}")
    else:
        scale = 1.0
        print(f"[{feature}] no normalisation (scale = 1.0)")

    for df, raw in ((df_trials, raw_trials), (df_targets, raw_targets)):
        df[f"{feature}_raw"] = raw
        df[feature] = raw.astype(np.float32) * scale

    return scale

In [ ]:
## run every feature

NORM_SCALES = {feature: add_feature(feature) for feature in FEATURE_NAMES}

display(df_trials[["png_filename", "condition_code"] + FEATURE_NAMES].head())
display(df_targets[["png_filename", "condition_code"] + FEATURE_NAMES].head())

In [ ]:
## save the feature tables
#
# Full rewrite. These files are regenerated from scratch every run, so they can
# never retain rows from a previous anonymisation scheme.

df_trials.to_csv(TRIALS_FEATURES_CSV, index=False)
df_targets.to_csv(TARGETS_FEATURES_CSV, index=False)

print(f"Saved {TRIALS_FEATURES_CSV.name}: {len(df_trials):,} rows")
print(f"Saved {TARGETS_FEATURES_CSV.name}: {len(df_targets):,} rows")
print("\nnorm scales:")
for k, v in NORM_SCALES.items():
    print(f"  {k:<22} {v:.6g}")

## Diagnostics

Nothing below this point writes to the feature tables. These cells are visual
checks plus the paper figures.

In [ ]:
## shared plotting helpers

def save_tight(fig, path, pad: float = 0.0):
    fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=pad)
    print(f"saved: {path}")


def bin_mean_map(map2d: np.ndarray, bins: int) -> np.ndarray:
    """Reduce a 2D map to (bins x bins) by taking the mean within each cell."""
    h, w = map2d.shape
    out = np.full((bins, bins), np.nan, dtype=np.float32)
    for i in range(bins):
        y0, y1 = int(i * h / bins), int((i + 1) * h / bins)
        for j in range(bins):
            x0, x1 = int(j * w / bins), int((j + 1) * w / bins)
            vals = map2d[y0:y1, x0:x1]
            vals = vals[np.isfinite(vals)]
            out[i, j] = float(vals.mean()) if vals.size else np.nan
    return out


def fmt_cell(v: float) -> str:
    """Two decimals with no leading zero; '1' when v is ~1."""
    if np.isfinite(v) and abs(v - 1.0) < 1e-6:
        return "1"
    s = f"{v:.2f}"
    return s[1:] if s.startswith("0") else s


def ceil_2dp(x: float) -> float:
    return math.ceil(x * 100.0) / 100.0 if np.isfinite(x) else 1.0


def condition_label(cond) -> str:
    """'1' -> '2.5 Hz'; anything else passes through unchanged."""
    ci = _condition_code_to_int(cond)
    return f"{_format_hz(HZ_MAP[ci])} Hz" if ci in HZ_MAP else str(cond)


def experimental_conditions(df: pd.DataFrame) -> list[str]:
    """Strobe conditions present in df, ordered by frequency."""
    vals = df["condition_code"].dropna().astype(str)
    uniq = [c for c in vals.unique() if c in EXPERIMENTAL_CONDS]
    return sorted(uniq, key=float)

In [ ]:
## sanity check: a few random images from each set

def show_random_images(df, img_dir, label, n=3, size=1.2, seed=None):
    if seed is not None:
        random.seed(seed)
    names = df["png_filename"].dropna().astype(str).tolist()
    sample = random.sample(names, min(n, len(names)))

    print(f"{label} — random image check")
    fig, axes = plt.subplots(1, len(sample), figsize=(size * len(sample), size))
    axes = np.atleast_1d(axes)
    for ax, fname in zip(axes, sample):
        ax.imshow(load_grayscale(img_dir / fname), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    plt.tight_layout(pad=0.1)
    plt.show()


show_random_images(df_trials, TRIAL_IMG_DIR, "VizEx trials")
show_random_images(df_targets, TARGET_IMG_DIR, "VizEx targets")

In [ ]:
## sanity check: original vs the two preprocessing pipelines

def show_preprocessing_check(df, img_dir, label, seed=None, figsize=(4.2, 1.6)):
    row = df.sample(1, random_state=seed).iloc[0]
    img = load_grayscale(img_dir / str(row["png_filename"]))

    print(f"{label} | file={row['png_filename']}")
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[1].imshow(preprocess_edges(img), cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(preprocess_intensity(img), cmap="gray", vmin=0, vmax=1)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout(pad=0.05)
    plt.show()


show_preprocessing_check(df_trials, TRIAL_IMG_DIR, "VizEx trials")
show_preprocessing_check(df_targets, TARGET_IMG_DIR, "VizEx targets")

In [ ]:
## single-image feature breakdown
#
# original | preprocessed | raw heatmap | 7x7 binned heatmap with values | colorbar

def show_feature_breakdown(
    df, img_dir, feature, *,
    label="VizEx",
    bins=7,
    display_mode="per_image",   # "per_image" or "normalized"
    seed=None,
    image_idx=None,
    figsize=(11.2, 2.2),
    cbar_width_ratio=0.18,
):
    spec = FEATURES[feature]
    norm_scale = NORM_SCALES[feature]

    row = df.loc[image_idx] if image_idx is not None else df.sample(1, random_state=seed).iloc[0]
    fname = str(row["png_filename"])

    orig = load_grayscale(img_dir / fname)
    pre = spec["preprocess"](orig)
    fmap = spec["map_fn"](pre)

    scalar_raw = spec["value_fn"](fmap)
    bmap = bin_mean_map(fmap, bins)

    if display_mode == "normalized":
        fmap_disp = np.clip(fmap * norm_scale, 0.0, 1.0)
        bmap_disp = np.clip(bmap * norm_scale, 0.0, 1.0)
        print(f"{label} | {feature} | condition={row.get('condition_code', '?')} | file={fname} "
              f"| raw={scalar_raw:.6f} | norm={scalar_raw * norm_scale:.6f} "
              f"| norm_scale={norm_scale:.6g}")
    elif display_mode == "per_image":
        vmax_bin = float(np.nanmax(bmap)) if np.isfinite(bmap).any() else 0.0
        if vmax_bin > 0:
            bmap_disp = (bmap / vmax_bin).astype(np.float32)
            fmap_disp = (fmap / vmax_bin).astype(np.float32)
        else:
            bmap_disp = np.zeros_like(bmap)
            fmap_disp = np.zeros_like(fmap)
        print(f"{label} | {feature} | condition={row.get('condition_code', '?')} | file={fname} "
              f"| raw={scalar_raw:.6f}")
    else:
        raise ValueError("display_mode must be 'per_image' or 'normalized'.")

    fig, axes = plt.subplots(
        1, 5, figsize=figsize,
        gridspec_kw={"width_ratios": [1, 1, 1, 1, cbar_width_ratio]},
    )
    axes[0].imshow(orig, cmap="gray", vmin=0, vmax=1)
    axes[1].imshow(pre, cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(fmap_disp, cmap="rainbow", vmin=0, vmax=1)
    im = axes[3].imshow(bmap_disp, cmap="rainbow", vmin=0, vmax=1,
                        interpolation="nearest", origin="upper")

    for i in range(bins):
        for j in range(bins):
            axes[3].text(j, i, fmt_cell(float(bmap_disp[i, j])),
                         ha="center", va="center", fontsize=8)

    for ax in axes[:4]:
        ax.axis("off")

    cbar = fig.colorbar(im, cax=axes[4], orientation="vertical")
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels([fmt_cell(0.0), "1"])
    cbar.outline.set_visible(False)
    plt.show()


def run_breakdown_diagnostics(feature, *, display_mode="per_image", n_trials=2):
    for _ in range(n_trials):
        show_feature_breakdown(df_trials, TRIAL_IMG_DIR, feature,
                               label="VizEx trials", display_mode=display_mode)
    show_feature_breakdown(df_targets, TARGET_IMG_DIR, feature,
                           label="VizEx targets", display_mode=display_mode)


run_breakdown_diagnostics("detail", display_mode="per_image")

In [ ]:
## normalised view of the same breakdown

run_breakdown_diagnostics("detail", display_mode="normalized")

In [ ]:
## orientation feature breakdowns

run_breakdown_diagnostics("r_directionality", display_mode="per_image")
run_breakdown_diagnostics("theta_directionality", display_mode="per_image")

In [ ]:
## feature distributions across the trials

def plot_feature_distribution(df, feature, *, label="VizEx", bins=40, figsize=(6, 3)):
    vals = df[feature].astype(float).dropna().values
    if vals.size == 0:
        print(f"[{label} | {feature}] no valid values, skipping.")
        return

    sd = float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0
    print(f"{label} | {feature} | min={vals.min():.4f}, mean={vals.mean():.4f}, "
          f"sd={sd:.4f}, max={vals.max():.4f} | norm_scale={NORM_SCALES[feature]:.6g}")

    fig, ax = plt.subplots(figsize=figsize)
    ax.hist(vals, bins=bins, edgecolor="black")
    ax.set_xlim(vals.min(), vals.max())
    ax.set_yticks([])
    ax.set_xlabel(f"{feature} (normalised)")
    ax.set_ylabel("Frequency")
    plt.show()


for _feature in FEATURE_NAMES:
    plot_feature_distribution(df_trials, _feature)

In [ ]:
## feature spectrum: images ordered low -> high on the stored scalar

def select_spectrum_rows(df, value_col, n, *, target_min=0.0, target_max=1.0,
                         jitter_frac=0.15, k_candidates=12, seed=None):
    """Pick n rows whose stored values span target_min -> target_max.

    Targets are jittered and one of the k nearest candidates is chosen at random,
    so repeated calls show different images from the same part of the range.
    """
    vals = pd.to_numeric(df[value_col], errors="coerce")
    keep = np.isfinite(vals.values)
    if not keep.any():
        return df.iloc[0:0]

    vals = vals.values[keep]
    idxs = df.index.values[keep]
    rng = np.random.default_rng(seed)

    targets = np.linspace(target_min, target_max, n) if n > 1 else np.array([(target_min + target_max) / 2])
    step = (target_max - target_min) / (n - 1) if n > 1 else 0.0

    used, chosen = set(), []
    for t in targets:
        t_j = float(t + rng.uniform(-jitter_frac * step, jitter_frac * step)) if step > 0 else float(t)
        diffs = np.abs(vals - t_j)
        if used:
            diffs = np.where(np.isin(idxs, list(used)), np.inf, diffs)

        n_finite = int(np.isfinite(diffs).sum())
        if n_finite == 0:
            continue
        k = min(max(1, k_candidates), n_finite)
        nearest = np.argpartition(diffs, kth=k - 1)[:k]
        nearest = nearest[np.isfinite(diffs[nearest])]
        if nearest.size == 0:
            continue

        pick = idxs[int(nearest[int(rng.integers(0, nearest.size))])]
        used.add(pick)
        chosen.append(pick)

    return df.loc[chosen]


def show_feature_spectrum(df, img_dir, feature, *, label="VizEx", n=11, bins=7,
                          figsize_per_col=1.5, cmap="rainbow", seed=None,
                          color_scale=None):
    """Four rows: original, preprocessed, normalised heatmap, binned heatmap.

    The colour scale is shared across every cell and every image. By default it
    runs 0 -> max(1, largest binned value), so values above 1 stay visible.
    """
    spec = FEATURES[feature]
    norm_scale = NORM_SCALES[feature]

    if color_scale is not None:
        vmin, vmax = float(color_scale[0]), float(color_scale[1])
        if not (np.isfinite(vmin) and np.isfinite(vmax)) or vmax <= vmin:
            raise ValueError("color_scale must be a finite (vmin, vmax) with vmax > vmin")
        target_min, target_max = vmin, vmax
    else:
        vmin = vmax = None
        target_min, target_max = 0.0, 1.0

    df_sel = select_spectrum_rows(df, feature, n, target_min=target_min,
                                  target_max=target_max, seed=seed)
    if df_sel.empty:
        print(f"{label} | {feature}: no valid values to build spectrum.")
        return

    df_sel = df_sel.sort_values(by=feature, ascending=True, kind="mergesort")

    items, global_max = [], 0.0
    for _, row in df_sel.iterrows():
        orig = load_grayscale(img_dir / str(row["png_filename"]))
        pre = spec["preprocess"](orig)
        raw_map = spec["map_fn"](pre)
        fmap = raw_map * float(norm_scale)
        bmap = bin_mean_map(raw_map, bins) * float(norm_scale)

        if np.isfinite(bmap).any():
            global_max = max(global_max, float(np.nanmax(bmap)))
        items.append({"orig": orig, "pre": pre, "fmap": fmap, "bmap": bmap,
                      "value": float(row[feature])})

    if color_scale is None:
        vmin, vmax = 0.0, max(1.0, global_max)

    print(f"{label} | {feature} spectrum | n={len(items)} | norm_scale={norm_scale:.6g} "
          f"| vmin={vmin:.6g} | vmax={vmax:.6g}")

    n_cols = len(items)
    fig, axes = plt.subplots(
        4, n_cols + 1,
        figsize=(n_cols * figsize_per_col + 1.0, 4.8),
        gridspec_kw={"width_ratios": [1] * n_cols + [0.20]},
    )
    axes = np.array(axes).reshape(4, n_cols + 1)

    for c, it in enumerate(items):
        axes[0, c].text(0.5, 1.05, f"{it['value']:.2f}".lstrip("0"),
                        transform=axes[0, c].transAxes,
                        ha="center", va="bottom", fontsize=10)
        axes[0, c].imshow(it["orig"], cmap="gray", vmin=0, vmax=1)
        axes[1, c].imshow(it["pre"], cmap="gray", vmin=0, vmax=1)
        axes[2, c].imshow(it["fmap"], cmap=cmap, vmin=vmin, vmax=vmax)
        im = axes[3, c].imshow(it["bmap"], cmap=cmap, vmin=vmin, vmax=vmax,
                               interpolation="nearest", origin="upper")
        for r in range(4):
            axes[r, c].axis("off")

    for r in range(4):
        axes[r, -1].axis("off")

    cax = fig.add_axes([
        axes[1, -1].get_position().x0,
        axes[3, -1].get_position().y0,
        axes[1, -1].get_position().width,
        axes[1, -1].get_position().y1 - axes[3, -1].get_position().y0,
    ])
    cbar = fig.colorbar(im, cax=cax, orientation="vertical")
    cbar.set_ticks([vmin, vmax])
    cbar.set_ticklabels([fmt_cell(vmin), fmt_cell(vmax)])
    cbar.outline.set_visible(False)
    plt.show()


for _feature in ["detail", "r_directionality", "theta_directionality", "brightness"]:
    show_feature_spectrum(df_trials, TRIAL_IMG_DIR, _feature, label="VizEx trials", n=11)

show_feature_spectrum(df_trials, TRIAL_IMG_DIR, "contrast", label="VizEx trials",
                      n=11, color_scale=(0.0, 0.5))

## Condition heatmaps

In [ ]:
## robust across-image reduction, and the scalar renderers

def robust_reduce(stack: np.ndarray, method: str = "trimmed_mean", trim_q: float = TRIM_Q) -> np.ndarray:
    """Reduce a (n_images, n_bins, n_bins) stack to (n_bins, n_bins)."""
    if stack.size == 0:
        raise ValueError("Empty stack passed to robust_reduce")
    if method == "median":
        return np.nanmedian(stack, axis=0)
    if method == "trimmed_mean":
        lo = np.nanquantile(stack, trim_q, axis=0)
        hi = np.nanquantile(stack, 1.0 - trim_q, axis=0)
        return np.nanmean(np.clip(stack, lo, hi), axis=0)
    raise ValueError(f"Unknown robust method: {method}")


def binned_heatmap_for_group(df_group, img_dir, *, preprocess_fn, map_fn,
                             n_bins=N_BINS, norm_scale=1.0, method="trimmed_mean"):
    """Bin each image's feature map, then robustly reduce across images."""
    binned = []
    for fname in df_group["png_filename"].astype(str):
        path = img_dir / fname
        if not path.exists():
            continue
        binned.append(bin_mean_map(map_fn(preprocess_fn(load_grayscale(path))), n_bins))

    if not binned:
        return np.full((n_bins, n_bins), np.nan, dtype=np.float32)

    out = robust_reduce(np.stack(binned, axis=0), method=method).astype(np.float32)
    return out * float(norm_scale)


def render_and_save_heatmap_rgb(rgb: np.ndarray, *, save_path, figsize=HEATMAP_FIGSIZE):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb, interpolation="nearest", origin="lower")
    ax.set_axis_off()
    ax.set_position([0, 0, 1, 1])
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    save_tight(fig, save_path, pad=0.0)
    plt.show()
    plt.close(fig)


def render_and_save_heatmap_scalar(hm: np.ndarray, *, vmin, vmax, cmap, save_path,
                                   figsize=HEATMAP_FIGSIZE):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(hm, interpolation="nearest", origin="lower", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_axis_off()
    ax.set_position([0, 0, 1, 1])
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    save_tight(fig, save_path, pad=0.0)
    plt.show()
    plt.close(fig)


def make_scale_bar(*, save_path, ylabel, rgb_column=None, cmap=None,
                   vmin=0.0, vmax=1.0, ticks=None, text_size=11, decimals=2):
    """Vertical scale bar. Supply either a 256x1x3 RGB column or a matplotlib cmap."""
    n = 256
    fig, ax = plt.subplots(figsize=SCALE_FIGSIZE)

    if rgb_column is not None:
        ax.imshow(rgb_column, aspect="auto", origin="upper")
    else:
        ax.imshow(np.linspace(1.0, 0.0, n).reshape(n, 1), aspect="auto",
                  cmap=cmap, origin="upper", vmin=0, vmax=1)

    ax.set_xticks([])
    ax.set_ylabel(ylabel, fontsize=text_size)

    if ticks is None:
        ticks = np.linspace(vmin, vmax, 5)
    ticks = np.clip(np.asarray(ticks, dtype=float), vmin, vmax)

    frac = (ticks - vmin) / (vmax - vmin) if vmax > vmin else np.zeros_like(ticks)
    ax.set_yticks((1.0 - frac) * (n - 1))
    ax.set_yticklabels([f"{t:.{decimals}f}" for t in ticks], fontsize=text_size)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    save_tight(fig, save_path, pad=0.05)
    plt.show()
    plt.close(fig)


def gradient_column(rgb_top) -> np.ndarray:
    """256x1x3 column fading from black at the bottom to rgb_top at the top."""
    t = np.linspace(1.0, 0.0, 256).reshape(256, 1)
    r, g, b = rgb_top
    return np.concatenate([t * r, t * g, t * b], axis=1).reshape(256, 1, 3)

In [ ]:
## directionality heatmaps (radial -> green, tangential -> red)

def directionality_heatmaps(df, img_dir, *, dataset_label="vizex",
                            n_bins=N_BINS, max_val_override=None):
    r_scale = NORM_SCALES["r_directionality"]
    t_scale = NORM_SCALES["theta_directionality"]

    hm_by_cond, max_val = {}, 0.0
    for cond in experimental_conditions(df):
        df_g = df[df["condition_code"].astype(str) == cond]
        if df_g.empty:
            continue

        gr = binned_heatmap_for_group(
            df_g, img_dir, preprocess_fn=preprocess_edges,
            map_fn=lambda p: sobel_radial_tangential_magnitude(p)[0],
            n_bins=n_bins, norm_scale=r_scale)
        gt = binned_heatmap_for_group(
            df_g, img_dir, preprocess_fn=preprocess_edges,
            map_fn=lambda p: sobel_radial_tangential_magnitude(p)[1],
            n_bins=n_bins, norm_scale=t_scale)

        hm_by_cond[cond] = (gr, gt)
        max_val = max(max_val, float(np.nanmax(gr)), float(np.nanmax(gt)))

    if max_val_override is not None:
        max_val = float(max_val_override)

    vmax = max(ceil_2dp(max_val), 0.01)
    print(f"{dataset_label} directionality vmax (shared across conditions) = {vmax:.2f}")

    saved = []
    for cond, (gr, gt) in hm_by_cond.items():
        print(f"{dataset_label} | condition={cond}")
        rgb = np.stack([
            np.clip(gt / vmax, 0, 1),   # tangential -> red
            np.clip(gr / vmax, 0, 1),   # radial     -> green
            np.zeros_like(gr),
        ], axis=-1)
        path = VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_{condition_label(cond)}_dir_heatmap.png"
        render_and_save_heatmap_rgb(rgb, save_path=path)
        saved.append(path)

    tick_step = max(ceil_2dp(vmax / 4.0), 0.01)
    n_steps = max(int(np.ceil(vmax / tick_step)), 1)
    ticks = np.linspace(0.0, vmax, n_steps + 1)

    make_scale_bar(save_path=VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_dir_scale_radial.png",
                   ylabel="Radial", rgb_column=gradient_column((0, 1, 0)),
                   vmin=0.0, vmax=vmax, ticks=ticks)
    make_scale_bar(save_path=VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_dir_scale_tangential.png",
                   ylabel="Tangential", rgb_column=gradient_column((1, 0, 0)),
                   vmin=0.0, vmax=vmax, ticks=ticks)
    return saved


dir_paths = directionality_heatmaps(df_trials, TRIAL_IMG_DIR)

In [ ]:
## detail heatmaps: one per condition, plus a pooled experimental map
#
# The pooled map builds one heatmap per participant and then takes the median
# across participants, so a participant with many trials cannot dominate.

def detail_heatmaps(df, img_dir, *, dataset_label="vizex", n_bins=N_BINS,
                    cmap="inferno", participant_col="participant_code",
                    pooled_method="median"):
    scale = NORM_SCALES["detail"]

    hm_by_cond, max_val = {}, 0.0
    for cond in experimental_conditions(df):
        df_g = df[df["condition_code"].astype(str) == cond]
        if df_g.empty:
            continue
        hm = binned_heatmap_for_group(df_g, img_dir, preprocess_fn=preprocess_edges,
                                      map_fn=detail_map, n_bins=n_bins, norm_scale=scale)
        hm_by_cond[cond] = hm
        max_val = max(max_val, float(np.nanmax(hm)))

    # Pooled across experimental conditions, excluding the control.
    if participant_col not in df.columns:
        raise KeyError(f"participant_col '{participant_col}' not in dataframe. "
                       f"Available: {sorted(df.columns.tolist())}")

    df_exp = df[
        df["condition_code"].astype(str).isin(EXPERIMENTAL_CONDS)
        & (df["condition_code"].astype(str) != CONTROL_CODE)
    ]
    if df_exp.empty:
        raise ValueError("No experimental rows after excluding the control condition.")

    per_participant = [
        binned_heatmap_for_group(df_p, img_dir, preprocess_fn=preprocess_edges,
                                 map_fn=detail_map, n_bins=n_bins, norm_scale=scale)
        for _, df_p in df_exp.groupby(participant_col) if not df_p.empty
    ]
    if not per_participant:
        raise ValueError("No participant heatmaps could be computed.")

    pooled = robust_reduce(np.stack(per_participant, axis=0), method=pooled_method)
    print(f"pooled over {len(per_participant)} participants ({pooled_method})")

    hm_by_cond[f"pooled_no_{CONTROL_CODE}"] = pooled
    max_val = max(max_val, float(np.nanmax(pooled)))

    vmax = max(ceil_2dp(max_val), 0.01)
    print(f"{dataset_label} detail vmax (shared across all outputs) = {vmax:.2f}")

    saved = []
    for cond, hm in hm_by_cond.items():
        print(f"{dataset_label} | condition={cond}")
        path = VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_{condition_label(cond)}_detail_heatmap.png"
        render_and_save_heatmap_scalar(hm, vmin=0.0, vmax=vmax, cmap=cmap, save_path=path)
        saved.append(path)

    make_scale_bar(save_path=VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_detail_scale.png",
                   ylabel="Detail", cmap=cmap, vmin=0.0, vmax=vmax)
    return saved


detail_paths = detail_heatmaps(df_trials, TRIAL_IMG_DIR)

In [ ]:
## brightness + contrast heatmaps
#
# One combined image per condition: value carries brightness, saturation carries
# RMS contrast, hue is fixed green. Grey cells are low contrast, green cells high.

def binned_brightness_contrast(df_group, img_dir, n_bins=N_BINS):
    """Per-bin brightness and RMS contrast, robustly reduced across images.

    Contrast here is LOCAL: deviation is measured from each bin's own mean, not
    from the whole-image mean used by the scalar `contrast` feature.
    """
    bright, contr = [], []
    for fname in df_group["png_filename"].astype(str):
        path = img_dir / fname
        if not path.exists():
            continue
        pre = preprocess_intensity(load_grayscale(path))
        h, w = pre.shape

        b_out = np.full((n_bins, n_bins), np.nan, dtype=np.float32)
        c_out = np.full((n_bins, n_bins), np.nan, dtype=np.float32)
        for i in range(n_bins):
            y0, y1 = int(i * h / n_bins), int((i + 1) * h / n_bins)
            for j in range(n_bins):
                x0, x1 = int(j * w / n_bins), int((j + 1) * w / n_bins)
                vals = pre[y0:y1, x0:x1]
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    continue
                mu = float(vals.mean())
                b_out[i, j] = mu
                c_out[i, j] = float(np.sqrt(np.mean((vals - mu) ** 2)))

        bright.append(b_out)
        contr.append(c_out)

    if not bright:
        nan_hm = np.full((n_bins, n_bins), np.nan, dtype=np.float32)
        return nan_hm, nan_hm

    return (robust_reduce(np.stack(bright, axis=0)).astype(np.float32),
            robust_reduce(np.stack(contr, axis=0)).astype(np.float32))


def hsv_green_rgb(S: np.ndarray, V: np.ndarray) -> np.ndarray:
    """Green-hue HSV -> RGB for two (n, n) arrays in [0, 1]."""
    H = 1.0 / 3.0
    out = np.zeros(S.shape + (3,), dtype=np.float32)
    for i in range(S.shape[0]):
        for j in range(S.shape[1]):
            out[i, j] = colorsys.hsv_to_rgb(H, float(S[i, j]), float(V[i, j]))
    return out


def linmap01(x: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    """Linear map to [0, 1]; non-finite entries become 0 so HSV stays valid."""
    x = np.asarray(x, dtype=np.float32)
    if not (np.isfinite(vmin) and np.isfinite(vmax)) or vmax <= vmin:
        return np.zeros_like(x, dtype=np.float32)
    y = np.clip((x - vmin) / (vmax - vmin), 0.0, 1.0)
    y[~np.isfinite(x)] = 0.0
    return y.astype(np.float32, copy=False)


def brightness_contrast_heatmaps(df, img_dir, *, dataset_label="vizex", n_bins=N_BINS,
                                 autoscale=False, vmax_contrast=0.5, suffix=""):
    """autoscale=False: fixed scales, brightness 0-1 and contrast 0-vmax_contrast.
    autoscale=True:  both scales span the min/max actually present in the data.
    """
    hm_by_cond = {}
    for cond in experimental_conditions(df):
        df_g = df[df["condition_code"].astype(str) == cond]
        if df_g.empty:
            continue
        hm_by_cond[cond] = binned_brightness_contrast(df_g, img_dir, n_bins)

    if not hm_by_cond:
        return []

    if autoscale:
        b_all = np.stack([v[0] for v in hm_by_cond.values()])
        c_all = np.stack([v[1] for v in hm_by_cond.values()])
        b_min, b_max = float(np.nanmin(b_all)), float(np.nanmax(b_all))
        c_min, c_max = float(np.nanmin(c_all)), float(np.nanmax(c_all))
    else:
        b_min, b_max = 0.0, 1.0
        c_min, c_max = 0.0, float(vmax_contrast)

    saved = []
    for cond, (b_hm, c_hm) in hm_by_cond.items():
        V = linmap01(b_hm, b_min, b_max)
        S = linmap01(c_hm, c_min, c_max)
        print(f"{dataset_label} brightness+contrast{suffix} | condition={cond} "
              f"| B:[{b_min:.4g},{b_max:.4g}] C:[{c_min:.4g},{c_max:.4g}]")
        path = (VIZ_PATH /
                f"{SAVE_PREFIX}_{dataset_label}_{condition_label(cond)}"
                f"_brightness_contrast_heatmap{suffix}.png")
        render_and_save_heatmap_rgb(hsv_green_rgb(S, V), save_path=path)
        saved.append(path)

    decimals = 3 if autoscale else 2
    b_ticks = None if autoscale else np.arange(0.0, 1.0 + 1e-9, 0.25)
    c_ticks = None if autoscale else np.arange(0.0, c_max + 1e-9, 0.10)

    make_scale_bar(
        save_path=VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_brightness_scale{suffix}.png",
        ylabel="Brightness", cmap="gray", vmin=b_min, vmax=b_max,
        ticks=b_ticks, decimals=decimals)

    # Contrast bar: at fixed V=0.5, saturation runs from grey to full green.
    S_col = np.linspace(1.0, 0.0, 256).reshape(256, 1)
    contrast_col = np.zeros((256, 1, 3), dtype=np.float32)
    for i in range(256):
        contrast_col[i, 0] = colorsys.hsv_to_rgb(1.0 / 3.0, float(S_col[i, 0]), 0.5)

    make_scale_bar(
        save_path=VIZ_PATH / f"{SAVE_PREFIX}_{dataset_label}_contrast_scale{suffix}.png",
        ylabel="Contrast", rgb_column=contrast_col,
        vmin=c_min, vmax=c_max, ticks=c_ticks, decimals=decimals)

    return saved


bc_paths = brightness_contrast_heatmaps(df_trials, TRIAL_IMG_DIR, vmax_contrast=0.5)

In [ ]:
## autoscaled alternative view of the same brightness + contrast maps

bc_paths_auto = brightness_contrast_heatmaps(
    df_trials, TRIAL_IMG_DIR, autoscale=True, suffix="_AUTOSCALED")

In [ ]:
## horizontal colour scale, for dense multi-panel figures

def horizontal_color_scale(*, vmin=0.0, vmax=1.0, tick_step=0.2, decimals=1,
                           cmap="inferno", figsize=(6, 1), border_lw=2.0,
                           tick_lw=2.0, tick_length=6, label_size=12):
    fig, ax = plt.subplots(figsize=figsize)
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=ax, orientation="horizontal")
    ticks = np.arange(vmin, vmax + 1e-9, tick_step)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f"{t:.{decimals}f}" for t in ticks])
    cbar.ax.tick_params(axis="x", which="both", width=tick_lw,
                        length=tick_length, labelsize=label_size)

    for spine in cbar.ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(border_lw)

    plt.tight_layout()
    plt.show()


horizontal_color_scale()